# Empirical analyses

This notebook organizes the Italy, Texas, LA-MRSA, vaccination, and figure analyses. Public inputs are included in `data/`; restricted Meta Colocation Maps are not. Put authorized mobility files in `data/` using the names documented in `data/README.md`. A section with missing inputs reports them and stops cleanly rather than fabricating values.

The next cell loads the reusable numerical implementation from `numerical_method.ipynb`.

In [ ]:
%run numerical_method.ipynb

from pathlib import Path
import pandas as pd

DATA = Path("data")
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)


def require_files(*names):
    paths = [DATA / name for name in names]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        print("Section skipped; missing:", ", ".join(missing))
        return None
    return paths


def read_labeled_matrix(path):
    frame = pd.read_csv(path, index_col=0)
    if frame.shape[0] != frame.shape[1] or list(frame.index.astype(str)) != list(frame.columns.astype(str)):
        raise ValueError(f"{path} must be a labeled square matrix with identical row/column order")
    return frame.index.astype(str).to_list(), validate_C(frame.to_numpy(float))

## Italy

The public province populations and modeled global air-passenger flows are included. The restricted input is `italy_colocation.csv` (Meta, ADM2/NUTS 3, week 13 of 2023). Colocation probabilities are converted using destination population,

$$C_{ij}\propto C^{\mathrm{coloc}}_{ij}n_j,$$

and then normalized to spectral radius one. Importation weights are rebuilt from the included air-passenger flows and the airport-to-province assignments used in the current vaccination analysis.

In [ ]:
italy_inputs = require_files("italy_colocation.csv", "italy_population.csv")
italy_results = None
if italy_inputs:
    italy_names, italy_colocation = read_labeled_matrix(italy_inputs[0])
    population = (
        pd.read_csv(italy_inputs[1]).set_index("stratum")
        .loc[italy_names, "population"].to_numpy(float)
    )
    C_italy = normalize_C(italy_colocation * population[np.newaxis, :])
    italy_results = complex_critical_points(C_italy)
    print(f"Recovered {len(italy_results)} retained critical points")

## Texas

The repository includes the Texas DSHS 2024–2025 county MMR coverage and JHU 2025 county measles burden. The mobility input `texas_colocation.csv` is restricted Meta data (county level, week 9 of 2025). The population vector and optional reference-partition crosswalk follow the interfaces in `data/README.md`.

Susceptibility is applied on the receiving-stratum (row) side, preserving the reproduction-operator convention. Loving County is `NR` in the DSHS source and remains missing; the code raises a clear error if an authorized mobility matrix includes it, rather than silently imputing coverage.

In [ ]:
tx_vaccination = pd.read_csv(DATA / "texas_vaccination.csv")
tx_cases = pd.read_csv(DATA / "texas_measles_cases.csv")
print(f"Public Texas inputs: {len(tx_vaccination)} counties; {tx_cases.cases_2025.sum()} reported cases in 2025")

texas_inputs = require_files("texas_colocation.csv", "texas_population.csv")
texas_results = None
if texas_inputs:
    texas_names, texas_colocation = read_labeled_matrix(texas_inputs[0])
    tx_population = (
        pd.read_csv(texas_inputs[1]).set_index("stratum")
        .loc[texas_names, "population"].to_numpy(float)
    )
    tx_coverage = (
        tx_vaccination.set_index("stratum")
        .loc[texas_names, "coverage"].to_numpy(float)
    )
    if np.any(~np.isfinite(tx_coverage)):
        missing = np.asarray(texas_names)[~np.isfinite(tx_coverage)]
        raise ValueError(f"MMR coverage is unreported for: {', '.join(missing)}")
    C_texas = normalize_C(
        np.diag(1.0 - tx_coverage)
        @ (texas_colocation * tx_population[np.newaxis, :])
    )
    texas_results = complex_critical_points(C_texas)
    print(f"Recovered {len(texas_results)} retained critical points")

## LA-MRSA

The included matrix is Table S1 of Porphyre et al. (2012). Its rows describe contacts made by members of population $i$ with population $j$. The paper's operator instead defines $K_{ij}$ as potentially infectious contacts generated in receiving group $i$ by an individual in source group $j$, so the table is transposed explicitly before computing

$$C=K/\rho(K).$$

Two published cells are censored as `<0.001`. The code uses the reported upper bound `0.001` explicitly and announces this approximation; the source strings remain unchanged in the CSV.

In [ ]:
mrsa_path = DATA / "la_mrsa_contact_matrix.csv"
source_contacts = pd.read_csv(mrsa_path, index_col=0, dtype=str)
if source_contacts.shape[0] != source_contacts.shape[1] or list(source_contacts.index) != list(source_contacts.columns):
    raise ValueError("LA-MRSA source table must have identical row/column group order")

censored = source_contacts.apply(lambda column: column.str.startswith("<")).to_numpy()
contacts_numeric = source_contacts.replace(r"^<0\.001$", "0.001", regex=True).astype(float)
if censored.any():
    print(f"Using the published upper bound 0.001 for {censored.sum()} censored contact values")

mrsa_names = source_contacts.index.to_list()
K = contacts_numeric.to_numpy().T  # receiving group i by infectious source group j
C_mrsa = normalize_C(K)
mrsa_results = complex_critical_points(C_mrsa)
print(f"Recovered {len(mrsa_results)} retained critical points")

## Vaccination analysis

With perfect instantaneous immunity, vaccinating $V_i$ of $n_i$ residents leaves $s_i=1-V_i/n_i$ susceptible and changes the operator by row scaling,

$$R_{ij}^{(\mathbf V)}=s_i(V_i)R_{ij}.$$

The importation-weighted epidemic probability is $p(r,\mathbf V)=\sum_iW_ip_i(r,\mathbf V)$ and effectiveness is $\mathcal E=1-p(r,\mathbf V)/p(r,\mathbf 0)$. Following the latest Italy analysis, a province is associated with a retained critical mode when $|\widetilde w_i|^2>0.1$; if several modes qualify, the one with the largest $|\mathrm{Im}\,r_c|$ is retained. The fragmented-criticality ranking is

$$\mathrm{CFI}_i(r)=\frac{W_i}{\sqrt{(r-\mathrm{Re}\,r_i^c)^2+(\mathrm{Im}\,r_i^c)^2}}.$$

The score intentionally does **not** contain a baseline epidemic-probability factor $p_i$. The comparison uses vaccine totals of 100,000, 1,000,000, and 5,000,000 doses; ranks the top 5–10 provinces; and compares importation-risk targeting, CFI targeting, and the 50-province importation baseline. Vaccine efficacy is one, matching the current Figure 3 vaccination code. Source scenarios are Mexico ($r=1.2,2.3$; direct flights), Botswana ($r=3.0$; direct plus one stop), South Africa ($r=3.33$; direct plus one stop), and China ($r=2.3$; direct flights).

In [ ]:
from matplotlib.ticker import PercentFormatter

VACCINE_SCENARIOS = [100_000, 1_000_000, 5_000_000]
VACCINE_EFFICACY = 1.0
MAX_INTERVENED_PROVINCES = 10
UNIFORM_TARGET_PROVINCES = 50
SOURCE_SCENARIOS = {
    "mexico_new_cfi_no_pi": {"country_code": "MX", "label": "Mexico", "risk_method": "direct", "r_values": [1.2, 2.3]},
    "botswana_new_cfi_no_pi": {"country_code": "BW", "label": "Botswana", "risk_method": "direct_plus_one_stop", "r_values": [3.0]},
    "south_africa_new_cfi_no_pi": {"country_code": "ZA", "label": "South Africa", "risk_method": "direct_plus_one_stop", "r_values": [3.33]},
    "china_new_cfi_no_pi": {"country_code": "CN", "label": "China", "risk_method": "direct", "r_values": [2.3]},
}
AIRPORT_TO_PROVINCE = {
    "AOI": "Ancona", "BRI": "Bari", "BLQ": "Bologna", "BDS": "Brindisi",
    "CAG": "Cagliari", "CTA": "Catania", "FLR": "Firenze", "GOA": "Genova",
    "LIN": "Milano", "MXP": "Milano", "BGY": "Bergamo", "NAP": "Napoli",
    "PMO": "Palermo", "PEG": "Perugia", "PSR": "Pescara", "REG": "Reggio Di Calabria",
    "RMI": "Rimini", "FCO": "Roma", "CIA": "Roma", "SUF": "Catanzaro",
    "TPS": "Trapani", "TRS": "Trieste", "TRN": "Torino", "TSF": "Treviso",
    "VCE": "Venezia", "VRN": "Verona", "OLB": "Sassari", "AHO": "Sassari",
    "PMF": "Parma", "FRL": "Forli' - Cesena", "VBS": "Brescia",
    "QSR": "Salerno", "FOG": "Foggia",
}


def vaccinated_C(C, vaccinated, population):
    C = validate_C(C)
    vaccinated = np.asarray(vaccinated, dtype=float)
    population = np.asarray(population, dtype=float)
    if np.any(population <= 0) or np.any(vaccinated < 0) or np.any(vaccinated > population):
        raise ValueError("vaccinated counts must lie between zero and population")
    return np.diag(1.0 - VACCINE_EFFICACY * vaccinated / population) @ C


def targeted_vaccine_counts(population, targets, total_vaccines):
    population = np.asarray(population, dtype=float)
    targets = np.asarray(targets, dtype=int)
    vaccinated = np.zeros_like(population)
    target_population = population[targets].sum()
    if target_population <= 0:
        raise ValueError("target population sum is zero")
    vaccinated[targets] = np.minimum(
        total_vaccines * population[targets] / target_population, population[targets]
    )
    return vaccinated


def weighted_epidemic_probability(C, r, importation_weights):
    weights = np.asarray(importation_weights, dtype=float)
    weights = weights / weights.sum()
    return float(weights @ epidemic_probability(C, r))


def campaign_effectiveness(C, r, targets, total_vaccines, population, importation_weights):
    baseline = weighted_epidemic_probability(C, r, importation_weights)
    vaccinated = targeted_vaccine_counts(population, targets, total_vaccines)
    after = weighted_epidemic_probability(vaccinated_C(C, vaccinated, population), r, importation_weights)
    return np.nan if baseline == 0 else 100.0 * (1.0 - after / baseline)


def critical_fragility_index(importation_weights, associated_critical_points, r):
    weights = np.asarray(importation_weights, dtype=float)
    points = np.asarray(associated_critical_points, dtype=complex)
    distance = np.sqrt((r - points.real)**2 + points.imag**2)
    return weights / np.maximum(distance, 1e-12)


def country_labeled_flows(annual_flows, airport_info):
    airports = airport_info[["NodeName", "CountryCode"]].drop_duplicates()
    return (annual_flows
        .merge(airports.rename(columns={"NodeName": "Origin", "CountryCode": "OriginCountry"}), on="Origin", how="left")
        .merge(airports.rename(columns={"NodeName": "Destination", "CountryCode": "DestinationCountry"}), on="Destination", how="left")
        .dropna(subset=["OriginCountry", "DestinationCountry"]))


def italy_airport_risk(origin_country, annual_flows, airport_info, method):
    flows = country_labeled_flows(annual_flows, airport_info)
    flows = flows[flows["Flow"] > 0].copy()
    direct = (flows[(flows.OriginCountry == origin_country) & (flows.DestinationCountry == "IT")]
              .groupby("Destination")["Flow"].sum().astype(float))
    if method == "direct":
        scores = direct
    elif method == "direct_plus_one_stop":
        italian_airports = airport_info.loc[airport_info.CountryCode == "IT", "NodeName"].dropna().unique()
        source_to_hub = (flows[(flows.OriginCountry == origin_country) & (flows.DestinationCountry != "IT")]
                         .groupby("Destination")["Flow"].sum().astype(float))
        hub_to_italy = flows[flows.DestinationCountry == "IT"]
        hub_out_total = flows.groupby("Origin")["Flow"].sum().astype(float)
        one_stop = pd.Series(0.0, index=italian_airports)
        for hub, source_flow in source_to_hub.items():
            onward = hub_to_italy[hub_to_italy.Origin == hub]
            denominator = float(hub_out_total.get(hub, 0.0))
            if denominator > 0 and not onward.empty:
                onward_score = onward.groupby("Destination")["Flow"].sum() * source_flow / denominator
                one_stop = one_stop.add(onward_score, fill_value=0.0)
        scores = direct.add(one_stop, fill_value=0.0)
    else:
        raise ValueError(f"unknown risk method: {method}")
    scores = scores[scores > 0]
    if scores.empty:
        raise ValueError(f"no Italy-bound passenger flow for {origin_country}")
    return scores


def province_importation_weights(source, method, province_names, annual_flows, airport_info):
    airport_scores = italy_airport_risk(source, annual_flows, airport_info, method)
    frame = airport_scores.rename("risk").rename_axis("airport").reset_index()
    frame["province"] = frame["airport"].map(AIRPORT_TO_PROVINCE)
    province_scores = frame.dropna(subset=["province"]).groupby("province")["risk"].sum()
    weights = province_scores.reindex(province_names, fill_value=0.0).to_numpy(float)
    if weights.sum() <= 0:
        raise ValueError(f"no mapped province risk for {source}")
    return weights / weights.sum()


def associate_provinces_with_critical_points(results, province_names, localization_squared_cutoff=0.1):
    rows = []
    for point in results:
        for index in np.where(point["localization"]**2 > localization_squared_cutoff)[0]:
            rows.append((province_names[index], point["r_c"]))
    associated = np.full(len(province_names), np.nan + 1j*np.nan, dtype=complex)
    if not rows:
        return associated
    table = pd.DataFrame(rows, columns=["province", "r_c"])
    table["imaginary_magnitude"] = table.r_c.map(lambda z: abs(z.imag))
    table = table.sort_values("imaginary_magnitude", ascending=False).drop_duplicates("province")
    lookup = table.set_index("province").r_c
    for index, name in enumerate(province_names):
        if name in lookup.index:
            associated[index] = lookup.loc[name]
    return associated


def plot_italy_vaccination(source_key, config, C, population, province_names, critical_points, annual_flows, airport_info):
    weights = province_importation_weights(config["country_code"], config["risk_method"], province_names, annual_flows, airport_info)
    associated = associate_provinces_with_critical_points(critical_points, province_names)
    valid = np.where(~np.isnan(associated.real) & ~np.isnan(associated.imag))[0]
    if len(valid) < UNIFORM_TARGET_PROVINCES:
        raise ValueError(f"only {len(valid)} provinces have an associated critical point")
    importation_ranking = valid[np.argsort(weights[valid])[::-1]]
    n_rows = len(config["r_values"])
    fig, axes = plt.subplots(n_rows, len(VACCINE_SCENARIOS), figsize=(11, 5 if n_rows > 1 else 2.9), dpi=300, squeeze=False)
    summary = []
    for row, r_value in enumerate(config["r_values"]):
        cfi = critical_fragility_index(weights[valid], associated[valid], r_value)
        cfi_ranking = valid[np.argsort(cfi)[::-1]]
        for column, total_vaccines in enumerate(VACCINE_SCENARIOS):
            uniform = campaign_effectiveness(C, r_value, importation_ranking[:UNIFORM_TARGET_PROVINCES], total_vaccines, population, weights)
            x = np.arange(5, MAX_INTERVENED_PROVINCES + 1)
            importation_values = [campaign_effectiveness(C, r_value, importation_ranking[:k], total_vaccines, population, weights) for k in x]
            cfi_values = [campaign_effectiveness(C, r_value, cfi_ranking[:k], total_vaccines, population, weights) for k in x]
            summary.append({"source": config["label"], "r": r_value, "vaccines": total_vaccines,
                            "best_importation": max(importation_values), "best_cfi": max(cfi_values), "uniform": uniform})
            ax = axes[row, column]
            line_importation, = ax.plot(x, importation_values, color="#4E79A7", lw=2.2)
            line_cfi, = ax.plot(x, cfi_values, color="#E15759", lw=3.0)
            line_uniform = ax.axhline(uniform, color="#59A14F", lw=1.6, ls="--")
            ax.spines[["top", "right"]].set_visible(False)
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=1))
            ax.set(xlim=(5, MAX_INTERVENED_PROVINCES), xticks=x, ylim=(0, None))
            if row == 0:
                ax.set_title(rf"$V_{{\mathrm{{tot}}}}={total_vaccines:,}$ vaccines")
            if column == 0:
                ax.text(-0.32, 0.5, rf"$r={r_value}$", transform=ax.transAxes, rotation=90, ha="center", va="center")
            if row == 0 and column == len(VACCINE_SCENARIOS) - 1:
                ax.legend([line_importation, line_cfi, line_uniform], ["Importation risk", "Critical fragility index", "Uniform"], frameon=False, fontsize=8)
    fig.supxlabel("Top-k vaccinated provinces")
    fig.supylabel("Effectiveness of vaccination policies (%)")
    fig.tight_layout()
    output = FIGURES / f"figure3_vaccination_strategy_comparison_{source_key}.pdf"
    fig.savefig(output, bbox_inches="tight", transparent=True)
    plt.close(fig)
    return pd.DataFrame(summary), output


italy_vaccination_outputs = {}
if italy_results is None:
    print("Italy vaccination comparison skipped: add the authorized data/italy_colocation.csv file.")
else:
    public_population = pd.read_csv(DATA / "italy_population.csv").set_index("stratum").loc[italy_names]
    province_names = public_population["province"].tolist()
    annual_flows = pd.read_csv(DATA / "raw/air_travel/AnnualPassengerFlows.csv")
    airport_info = pd.read_csv(DATA / "raw/air_travel/AirportInfoWithCountry.csv")
    for source_key, config in SOURCE_SCENARIOS.items():
        summary, output = plot_italy_vaccination(source_key, config, C_italy, population, province_names, italy_results, annual_flows, airport_info)
        italy_vaccination_outputs[source_key] = {"summary": summary, "figure": output}
        print(f"Generated {output}")

## Figure generation

The compact helper below plots retained critical points and mode strength. It writes only generated outputs to `figures/`. Paper panels requiring restricted or other source data are produced only after those inputs have been supplied.

In [ ]:
def plot_critical_landscape(results, title, output_name=None):
    if not results:
        print(f"{title}: no results to plot")
        return
    points = np.array([item["r_c"] for item in results])
    strength = np.array([item["localization"].max() for item in results])
    fig, ax = plt.subplots(figsize=(5.2, 4.0))
    scatter = ax.scatter(points.real, points.imag, c=strength, cmap="viridis", vmin=0, vmax=1)
    ax.set(xlabel=r"$\mathrm{Re}(r_c)$", ylabel=r"$\mathrm{Im}(r_c)$", title=title)
    fig.colorbar(scatter, ax=ax, label="maximum mode localization")
    fig.tight_layout()
    if output_name:
        fig.savefig(FIGURES / output_name, dpi=300, bbox_inches="tight")
    plt.show()


plot_critical_landscape(italy_results, "Italy", "italy_critical_landscape.png")
plot_critical_landscape(texas_results, "Texas", "texas_critical_landscape.png")
plot_critical_landscape(mrsa_results, "LA-MRSA", "la_mrsa_critical_landscape.png")